In [ ]:
# LiitLLM — evaluation
#
# The headline question: does filtering a corpus FOR code-switching produce a
# model that code-switches? Both arms are scored with the same scorer that
# built the corpora, on identical prompts.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "no GPU — set the accelerator to T4 x2"
cap = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
print(f"{name}  sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), (
    f"{name} is sm_{cap[0]}{cap[1]}; Kaggle's PyTorch needs sm_70+. "
    "Set --accelerator NvidiaTeslaT4 (the default P100 will not work)."
)

In [ ]:
import glob
from pathlib import Path

def find_one(slug, filename):
    """Locate a mounted dataset or kernel output by slug, not a bare glob.

    Two things this has to get right:

    1. Kaggle mounts sources at /kaggle/input/datasets/<owner>/<slug>/... and
       /kaggle/input/kernels/<owner>/<slug>/..., NOT at /kaggle/input/<slug>/.
       The older flat layout is what most examples show, and a glob written for
       it silently matches nothing. Hence the leading **, which spans either.
    2. Every kernel output carries a full copy of the repo, so a glob for a bare
       filename finds stale snapshots from earlier runs. Anchoring on the slug
       and asserting exactly one match makes that impossible, not just unlikely.
    """
    hits = glob.glob(f"/kaggle/input/**/{slug}/**/{filename}", recursive=True)
    assert len(hits) == 1, (
        f"expected exactly 1 {filename} under a source named {slug!r}, found {hits}.\n"
        f"Sources actually mounted: {sorted(glob.glob('/kaggle/input/*/*/*'))}"
    )
    return Path(hits[0])

In [ ]:
import shutil, os, sys

REPO_SLUG = "liitllm-repo"   # dataset holding this repo
PREP_SLUG = "00-prep"        # kernel whose OUTPUT holds the corpora + tokenizer

repo_src = find_one(REPO_SLUG, "pyproject.toml").parent
REPO = Path("/kaggle/working/liitllm-repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(repo_src, REPO)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f"repo: {repo_src} -> {REPO}")

In [ ]:
BASELINE = 'liitllm-ckpt-baseline'
ABLATION = 'liitllm-ckpt-ablation'

In [ ]:
from liitllm.evaluate import compare
import glob
def ckpt(slug):
    hits = glob.glob(f'/kaggle/input/{slug}/**/ckpt.pt', recursive=True)
    assert len(hits) == 1, f'expected 1 ckpt.pt under {slug}, found {hits}'
    return hits[0]
tok = str(find_one(DATA_SLUG, 'tokenizer.json'))
compare(ckpt(BASELINE), ckpt(ABLATION), tok, out_dir='/kaggle/working/results')